# Evolution Strategies: CMA-ES, OpenAI ES, and PSO

This notebook implements and verifies zeroth-order continuous optimization algorithms: **OpenAI Antithetic ES**, **CMA-ES (Rank-mu Covariance Adaptation)**, and **Particle Swarm Optimization (PSO)** across numerical parity against golden checkpoints.

## Mathematical Formulations

1. **Antithetic OpenAI ES**: $\nabla_\theta J(\theta) \approx \frac{1}{2 P \sigma} \sum_{i=1}^P (F(\theta + \sigma \epsilon_i) - F(\theta - \sigma \epsilon_i)) \epsilon_i$
2. **CMA-ES Rank-$\mu$ Update**: $\mu^{(t+1)} = \sum_i w_i x_i, \quad C^{(t+1)} = (1 - c_{\text{cov}}) C^{(t)} + c_{\text{cov}} \sum_i w_i (x_i - \mu)(x_i - \mu)^T$
3. **PSO Kinematics**: $v^{(t+1)} = w v^{(t)} + c_1 r_1 \odot (p_{\text{best}} - x^{(t)}) + c_2 r_2 \odot (g_{\text{best}} - x^{(t)})$

In [ ]:
from pathlib import Path
import torch, time
from safetensors.torch import load_file

repo_root = Path.cwd()
while not (repo_root / "Cargo.toml").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
w_path = repo_root / "weights/09_causal_ai_and_evolutionary_search/04_evolution_strategies_cma_es_openai_es_and_pso.safetensors"
weights = load_file(str(w_path))
print(f"Loaded {len(weights)} tensors from {w_path.name}")

In [ ]:
def openai_es_grad(epsilons, pos_fit, neg_fit, sigma=0.1):
    p = epsilons.shape[0]
    diff = (pos_fit - neg_fit).unsqueeze(-1)
    return torch.sum(diff * epsilons, dim=0) / (2.0 * float(p) * sigma)

def cma_es_step(mu_old, cov_old, pop, weights_elite, c_cov=0.2):
    w = weights_elite.unsqueeze(-1)
    mu_new = torch.sum(w * pop, dim=0)
    y = pop - mu_old.unsqueeze(0)
    c_mu = y.t() @ (w * y)
    cov_new = (1.0 - c_cov) * cov_old + c_cov * c_mu
    return mu_new, cov_new

def pso_step(x, v, p_best, g_best, r1, r2, w_inertia=0.7, c1=1.4, c2=1.4):
    v_next = w_inertia * v + c1 * r1 * (p_best - x) + c2 * r2 * (g_best - x)
    return v_next, x + v_next

In [ ]:
# Parity verification against golden reference tensors
es_g = openai_es_grad(weights["es_epsilons"], weights["es_pos_fit"], weights["es_neg_fit"], 0.1)
assert torch.max(torch.abs(es_g - weights["es_grad_ref"])) < 1e-5, "ES mismatch"

mu_n, cov_n = cma_es_step(weights["cma_mu_old"], weights["cma_cov_old"], weights["cma_pop"], weights["cma_weights"], 0.2)
assert torch.max(torch.abs(mu_n - weights["cma_mu_new_ref"])) < 1e-5, "CMA-ES mu mismatch"
assert torch.max(torch.abs(cov_n - weights["cma_cov_new_ref"])) < 1e-5, "CMA-ES cov mismatch"

pv, px = pso_step(weights["pso_x"], weights["pso_v"], weights["pso_pbest"], weights["pso_gbest"], weights["pso_r1"], weights["pso_r2"], 0.7, 1.4, 1.4)
assert torch.max(torch.abs(pv - weights["pso_v_next_ref"])) < 1e-5, "PSO v mismatch"
assert torch.max(torch.abs(px - weights["pso_x_next_ref"])) < 1e-5, "PSO x mismatch"
print("All parity assertions passed with error < 1e-5!")

In [ ]:
# Benchmarking vectorized step execution
t0 = time.perf_counter()
for _ in range(500):
    _ = openai_es_grad(weights["es_epsilons"], weights["es_pos_fit"], weights["es_neg_fit"], 0.1)
    _ = cma_es_step(weights["cma_mu_old"], weights["cma_cov_old"], weights["cma_pop"], weights["cma_weights"], 0.2)
    _ = pso_step(weights["pso_x"], weights["pso_v"], weights["pso_pbest"], weights["pso_gbest"], weights["pso_r1"], weights["pso_r2"], 0.7, 1.4, 1.4)
t_bench = (time.perf_counter() - t0) / 500 * 1e6
print(f"Average fused ES+CMA+PSO iteration latency: {t_bench:.2f} us")

In [ ]:
# Failure Mode Demonstration: Antithetic vs Non-Antithetic Monte Carlo Variance
torch.manual_seed(42)
# True objective: F(theta) = -0.5 * ||theta - theta*||^2, true grad = -(theta - theta*)
theta = torch.tensor([1.0, -2.0, 3.0, -1.0])
theta_star = torch.tensor([0.0, 0.0, 0.0, 0.0])
true_grad = -(theta - theta_star)
sigma, p_samples = 0.1, 10
eps = torch.randn(p_samples, 4)
# Antithetic estimator
f_pos = -0.5 * torch.sum((theta.unsqueeze(0) + sigma * eps - theta_star)**2, dim=1)
f_neg = -0.5 * torch.sum((theta.unsqueeze(0) - sigma * eps - theta_star)**2, dim=1)
g_anti = openai_es_grad(eps, f_pos, f_neg, sigma=sigma)
# Non-antithetic one-sided estimator: (F(theta + sigma*eps) - F(theta)) / (P * sigma) * eps
f_0 = -0.5 * torch.sum((theta - theta_star)**2)
g_onesided = torch.sum((f_pos - f_0).unsqueeze(-1) * eps, dim=0) / (float(p_samples) * sigma)
err_anti = float(torch.norm(g_anti - true_grad))
err_onesided = float(torch.norm(g_onesided - true_grad))
print(f"Antithetic Error: {err_anti:.4f} vs One-Sided Error: {err_onesided:.4f}")
assert err_anti < err_onesided, "Antithetic estimator must achieve lower variance than one-sided"

## Key Takeaways & Navigation Links

- **OpenAI Antithetic ES** achieves scalable parallel zero-order optimization with minimal inter-worker communication.
- **CMA-ES** constructs a full second-order inverse Hessian metric through rank-$\mu$ covariance accumulation.
- **PSO** balances exploration and exploitation through inertia, cognitive, and social attractors.

### Navigation Links
- [Previous: Domain Generalization IRM GroupDRO V-REx](03_domain_generalization_irm_groupdro_vrex_coral_and_dann.ipynb)
- [Next: Quality Diversity MAP-Elites and Differentiable QD](05_quality_diversity_map_elites_and_differentiable_qd.ipynb)